# HR Turnover Analysis

**Question:** Which employees are most likely to leave for reasons the company could have prevented, and where should retention efforts focus?

**Data:** [Human Resources Data Set](https://www.kaggle.com/datasets/rhuebner/human-resources-data-set) (v14) by Dr. Rich Huebner and Dr. Carla Patalano, a synthetic dataset of 311 employees. The names in it are fictional.

**Approach:**
- Sort every exit into *preventable* (another job, pay, hours, unhappiness, career change), *unpreventable* (retiring, relocation, school, military, medical, maternity), or *fired*.
- Compare employees who stayed with those who left for preventable reasons, broken down by department, recruitment source, manager, and tenure.
- Export a cleaned table for the Power BI dashboard (`hr_turnover_dashboard.pbix`).

## Load the data
Read the HR dataset, trim stray whitespace from every text column, and take a first look at its shape, columns, and sample rows.

In [3]:
import pandas as pd

df = pd.read_csv("HRDataset_v14.csv")
# Remove stray spaces at the start and end of every text column
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].str.strip()
    
print(df.shape)
print(df.columns.tolist())
df.head()

(311, 36)
['Employee_Name', 'EmpID', 'MarriedID', 'MaritalStatusID', 'GenderID', 'EmpStatusID', 'DeptID', 'PerfScoreID', 'FromDiversityJobFairID', 'Salary', 'Termd', 'PositionID', 'Position', 'State', 'Zip', 'DOB', 'Sex', 'MaritalDesc', 'CitizenDesc', 'HispanicLatino', 'RaceDesc', 'DateofHire', 'DateofTermination', 'TermReason', 'EmploymentStatus', 'Department', 'ManagerName', 'ManagerID', 'RecruitmentSource', 'PerformanceScore', 'EngagementSurvey', 'EmpSatisfaction', 'SpecialProjectsCount', 'LastPerformanceReview_Date', 'DaysLateLast30', 'Absences']


,Employee_Name,EmpID,MarriedID,MaritalStatusID,GenderID,EmpStatusID,DeptID,PerfScoreID,FromDiversityJobFairID,Salary,...,ManagerName,ManagerID,RecruitmentSource,PerformanceScore,EngagementSurvey,EmpSatisfaction,SpecialProjectsCount,LastPerformanceReview_Date,DaysLateLast30,Absences
0,"Adinolfi, Wilson K",10026,0,0,1,1,5,4,0,62506,...,Michael Albert,22.0,LinkedIn,Exceeds,4.60,5,0,1/17/2019,0,1
1,"Ait Sidi, Karthikeyan",10084,1,1,1,5,3,3,0,104437,...,Simon Roup,4.0,Indeed,Fully Meets,4.96,3,6,2/24/2016,0,17
2,"Akinkuolie, Sarah",10196,1,1,0,5,5,3,0,64955,...,Kissy Sullivan,20.0,LinkedIn,Fully Meets,3.02,3,0,5/15/2012,0,3
3,"Alagbe,Trina",10088,1,1,0,1,5,3,0,64991,...,Elijiah Gray,16.0,Indeed,Fully Meets,4.84,5,0,1/3/2019,0,15
4,"Anderson, Carol",10069,0,2,0,5,5,3,0,50825,...,Webster Butler,39.0,Google Search,Fully Meets,5.00,4,0,2/1/2016,0,2


## Remove identifying columns and review exits
Drop names and zip codes for privacy (keeping `EmpID` as an anonymous key), then count employees by employment status and by termination reason.

In [4]:
# Privacy first: drop columns that identify people but aren't needed for analysis.
# EmpID stays as an anonymous key so we can still tell rows apart.
df = df.drop(columns=["Employee_Name", "Zip"])

# Who left, and how?
print(df["EmploymentStatus"].value_counts())
print()

# Why did they leave?
print(df["TermReason"].value_counts())

EmploymentStatus
Active                    207
Voluntarily Terminated     88
Terminated for Cause       16
Name: count, dtype: int64

TermReason
N/A-StillEmployed                   207
Another position                     20
unhappy                              14
more money                           11
career change                         9
hours                                 8
attendance                            7
return to school                      5
relocation out of area                5
performance                           4
military                              4
retiring                              4
no-call, no-show                      4
maternity leave - did not return      3
medical issues                        3
Learned that he is a gangster         1
Fatal attraction                      1
gross misconduct                      1
Name: count, dtype: int64


## Termination reasons by exit type
Filter to employees who left and cross-tabulate each termination reason against whether the exit was voluntary or for cause.

In [5]:
# Keep only people who left
leavers = df[df["EmploymentStatus"] != "Active"]

# Count each reason by type of exit
pd.crosstab(leavers["TermReason"], leavers["EmploymentStatus"])

EmploymentStatus,Terminated for Cause,Voluntarily Terminated
TermReason,,
Another position,0,20
Fatal attraction,1,0
Learned that he is a gangster,1,0
attendance,6,1
career change,0,9
gross misconduct,1,0
hours,0,8
maternity leave - did not return,0,3
medical issues,0,3


## Classify exits
Group voluntary termination reasons into preventable (e.g. another position, pay, hours) and unpreventable (e.g. retiring, relocation), then label every employee with an `ExitType`: Stayed, Fired, Preventable, Unpreventable, or Other.

In [6]:
# Voluntary exit reasons the company could have influenced vs. ones it couldn't
preventable = ["Another position", "unhappy", "more money", "hours", "career change"]
not_preventable = [
    "return to school",
    "relocation out of area",
    "retiring",
    "military",
    "maternity leave - did not return",
    "medical issues",
]


# Label one employee's row with how (or whether) they left
def exit_type(row):
    if row["EmploymentStatus"] == "Active":
        return "Stayed"
    if row["EmploymentStatus"] == "Terminated for Cause":
        return "Fired"
    if row["TermReason"] in preventable:
        return "Preventable exit"
    if row["TermReason"] in not_preventable:
        return "Unpreventable exit"
    return "Other exit"


# Apply the label to every row
df["ExitType"] = df.apply(exit_type, axis=1)
df["ExitType"].value_counts()

ExitType
Stayed                207
Preventable exit       62
Unpreventable exit     24
Fired                  16
Other exit              2
Name: count, dtype: int64

## Calculate tenure
Convert date columns to datetimes, find the data snapshot date (the latest date in the dataset), and compute `TenureYears`: hire to termination for leavers, hire to snapshot for current staff.

In [7]:
#convert date columns to actual dates
df["DateofHire"] = pd.to_datetime(df["DateofHire"])
df["DateofTermination"] = pd.to_datetime(df["DateofTermination"])
reviews = pd.to_datetime(df["LastPerformanceReview_Date"])

# The data was pulled at some point in the past, so find the latest date in it
snapshot = max(df["DateofTermination"].max(), reviews.max())
print("Data snapshot date:", snapshot.date())

# Leavers: termination date minus hire date. Still employed: snapshot minus hire date.
end_date = df["DateofTermination"].fillna(snapshot)
df["TenureYears"] = (end_date - df["DateofHire"]).dt.days / 365.25

# Group tenure into bands; TenureOrder lets Power BI sort the bands correctly
bands = ["Under 1 yr", "1-2 yrs", "2-3 yrs", "3-5 yrs", "5+ yrs"]
df["TenureBand"] = pd.cut(df["TenureYears"], bins=[0, 1, 2, 3, 5, float("inf")], labels=bands, right=False)
df["TenureOrder"] = df["TenureBand"].cat.codes
df["TenureBand"] = df["TenureBand"].astype(str)

df["TenureYears"].describe()

Data snapshot date: 2019-02-28


count    311.000000
mean       4.824348
std        2.193374
min        0.071184
25%        3.711157
50%        4.646133
75%        6.243669
max       13.136208
Name: TenureYears, dtype: float64

## Build the analysis set
Keep only employees who stayed or left for a preventable reason, add a 0/1 `PreventableExit` flag, and report the overall preventable turnover rate.

In [8]:
# Keep only the two groups we're comparing
analysis = df[df["ExitType"].isin(["Stayed", "Preventable exit"])].copy()

# 1 = left for a preventable reason, 0 = stayed
analysis["PreventableExit"] = (analysis["ExitType"] == "Preventable exit").astype(int)

print(len(analysis), "employees in the analysis")
print(f"Preventable turnover rate: {analysis['PreventableExit'].mean():.1%}")

269 employees in the analysis
Preventable turnover rate: 23.0%


## Turnover rate by department
Define a `rate_by` helper that returns the preventable turnover rate, number who left, and headcount for each group in a column, then apply it to `Department`. Groups with fewer than 5 employees are hidden, because a rate based on one or two people is misleading.

In [9]:
# Preventable turnover rate, number who left, and headcount per group, highest rate first.
# The mean of a 0/1 column is the share of 1s, i.e. the turnover rate.
def rate_by(column, data=analysis, min_employees=5):
    table = data.groupby(column)["PreventableExit"].agg(["mean", "sum", "count"])
    table.columns = ["Rate", "Left", "Employees"]
    table = table[table["Employees"] >= min_employees]
    return table.sort_values("Rate", ascending=False)


rate_by("Department")

,Rate,Left,Employees
Department,,,
Production,0.300000,54,180
Software Engineering,0.222222,2,9
Admin Offices,0.125000,1,8
IT/IS,0.090909,4,44
Sales,0.037037,1,27


## Turnover rate by recruitment source
Compare preventable turnover across the channels employees were hired through.

In [10]:
rate_by("RecruitmentSource")

,Rate,Left,Employees
RecruitmentSource,,,
Google Search,0.500000,19,38
Diversity Job Fair,0.480000,12,25
CareerBuilder,0.333333,6,18
Indeed,0.153846,12,78
LinkedIn,0.147059,10,68
Website,0.076923,1,13
Employee Referral,0.037037,1,27


## Turnover rate by manager
Compare preventable turnover across managers.

In [11]:
rate_by("ManagerName")

,Rate,Left,Employees
ManagerName,,,
Amy Dunn,0.529412,9,17
Webster Butler,0.466667,7,15
Kissy Sullivan,0.411765,7,17
Elijiah Gray,0.300000,6,20
Michael Albert,0.277778,5,18
Simon Roup,0.250000,3,12
David Stanley,0.250000,5,20
Brannon Miller,0.238095,5,21
Janet King,0.235294,4,17


## Focus on Production
Production has the highest turnover, so repeat the manager and recruitment-source breakdowns within that department only.

In [12]:
# Production only, so manager/source differences aren't just department differences
production = analysis[analysis["Department"] == "Production"]

print(rate_by("ManagerName", production))
print()
print(rate_by("RecruitmentSource", production))

                    Rate  Left  Employees
ManagerName                              
Amy Dunn        0.529412     9         17
Webster Butler  0.466667     7         15
Kissy Sullivan  0.411765     7         17
Elijiah Gray    0.300000     6         20
Michael Albert  0.277778     5         18
David Stanley   0.250000     5         20
Brannon Miller  0.238095     5         21
Janet King      0.230769     3         13
Kelley Spirea   0.200000     4         20
Ketsia Liebig   0.157895     3         19

                        Rate  Left  Employees
RecruitmentSource                            
Diversity Job Fair  0.571429     8         14
Google Search       0.514286    18         35
CareerBuilder       0.461538     6         13
Indeed              0.255814    11         43
LinkedIn            0.145455     8         55
Employee Referral   0.066667     1         15


## Turnover rate by tenure
Compare preventable turnover across tenure bands. Leavers' tenure stops at their exit date, so short bands naturally hold more leavers. Read this as *when* people tend to leave, not as proof that new hires are riskier.

In [13]:
# Sort by band (shortest to longest) instead of by rate so the trend is easy to read
rate_by(["TenureOrder", "TenureBand"]).sort_index()

,,Rate,Left,Employees
TenureOrder,TenureBand,,,
0,Under 1 yr,0.800000,4,5
1,1-2 yrs,0.888889,16,18
2,2-3 yrs,0.333333,8,24
3,3-5 yrs,0.215054,20,93
4,5+ yrs,0.108527,14,129


## Key findings

- **Preventable turnover is 23%**: 62 of 269 employees left for reasons the company could have influenced, most often another job, unhappiness, or pay.
- **Production is where the problem is.** It has a 30% preventable turnover rate, and 54 of the 62 preventable exits happened there.
- **Three Production managers stand out.** Amy Dunn (53%), Webster Butler (47%), and Kissy Sullivan (41%) have well above-average turnover, even compared with other Production managers.
- **Recruitment source matters.** Within Production, hires from Diversity Job Fair (57%), Google Search (51%), and CareerBuilder (46%) leave far more often than LinkedIn (15%) or Employee Referral (7%) hires.
- **Most preventable exits happen early.** Rates fall steadily with tenure, from above 80% in the first two years to 11% after five years. Because leavers' tenure ends when they leave, this shows *when* people leave, not that new hires are inherently riskier.

**Where to focus:** onboarding and first-two-year check-ins in Production, a closer look at the three high-turnover teams, and shifting hiring toward referrals and LinkedIn.

## Export for Power BI

In [14]:
export = df.copy()
export["PreventableExit"] = (export["ExitType"] == "Preventable exit").astype(int)
export["InAnalysis"] = export["ExitType"].isin(["Stayed", "Preventable exit"]).astype(int)

columns = [
    "EmpID", "Department", "Position", "RecruitmentSource", "EmploymentStatus",
    "TermReason", "ExitType", "PreventableExit", "InAnalysis", "TenureYears",
    "TenureBand", "TenureOrder", "EngagementSurvey", "EmpSatisfaction", "Salary",
    "Absences", "DaysLateLast30", "PerformanceScore",
]
export[columns].to_csv("hr_turnover_for_powerbi.csv", index=False)
print(len(export), "rows written to hr_turnover_for_powerbi.csv")

311 rows written to hr_turnover_for_powerbi.csv
